# 02 - Data Validation

This notebook performs basic validation checks on the processed UK electricity demand dataset.

The validation workflow includes:

- Checking that the processed file exists
- Loading the processed dataset
- Validating row counts
- Checking for missing values
- Checking for duplicate datetime records
- Checking for invalid demand values

In [2]:
%reset -f

import pandas as pd
from pathlib import Path

## Load Processed Dataset

In [5]:
processed_file = Path("../Data/Processed/combined_energy_demand.csv")

df = pd.read_csv(processed_file)
df.head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,...,BRITNED_FLOW,MOYLE_FLOW,EAST_WEST_FLOW,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,SCOTTISH_TRANSFER,DATETIME
0,2021-01-01,1,28354,28969,26130,1158,6527,0,13471,0,...,0,215,203,999,0,0,0,0,0.0,2021-01-01 00:00:00
1,2021-01-01,2,28501,29114,26281,1208,6527,0,13471,0,...,0,359,203,999,0,0,0,0,0.0,2021-01-01 00:30:00
2,2021-01-01,3,27759,28376,25557,1202,6527,0,13471,0,...,0,362,202,999,0,0,0,0,0.0,2021-01-01 01:00:00
3,2021-01-01,4,26912,27749,24792,1226,6527,0,13471,0,...,0,361,203,1000,0,0,0,0,0.0,2021-01-01 01:30:00
4,2021-01-01,5,26004,27178,23933,1193,6527,0,13471,0,...,0,304,203,1000,0,0,0,0,0.0,2021-01-01 02:00:00


## Validate Dataset Shape

In [8]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 89,472
Columns: 23


## Validate Missing Values

In [11]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0]

Series([], dtype: int64)

## Validate Duplicate Datetime Records

In [14]:
duplicate_datetimes = df["DATETIME"].duplicated().sum()
print(f"Duplicate datetime records: {duplicate_datetimes}")

Duplicate datetime records: 10


## Inspect Duplicate Datetime Records

In [17]:
duplicate_rows = df[
    df["DATETIME"].duplicated(keep=False)
].sort_values("DATETIME")

duplicate_rows[
    ["SETTLEMENT_DATE", "SETTLEMENT_PERIOD", "DATETIME"]
]

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,DATETIME
14590,2021-10-31,49,2021-11-01 00:00:00
14592,2021-11-01,1,2021-11-01 00:00:00
14591,2021-10-31,50,2021-11-01 00:30:00
14593,2021-11-01,2,2021-11-01 00:30:00
32062,2022-10-30,49,2022-10-31 00:00:00
32064,2022-10-31,1,2022-10-31 00:00:00
32063,2022-10-30,50,2022-10-31 00:30:00
32065,2022-10-31,2,2022-10-31 00:30:00
49534,2023-10-29,49,2023-10-30 00:00:00
49536,2023-10-30,1,2023-10-30 00:00:00


## Duplicate Datetime Observation

Duplicate datetime records were identified during validation.

Inspection shows that these duplicates occur during daylight saving time transitions, where settlement periods 49 and 50 overlap with the following day's initial settlement periods.

These records are considered valid and will be retained in the dataset.

## Validate Negative Demand Values

In [21]:
negative_demand = df[df["ENGLAND_WALES_DEMAND"] < 0]

print(f"Negative demand records: {len(negative_demand)}")

Negative demand records: 0


## Validation Summary

The processed dataset successfully passed the primary validation checks:

- Processed dataset loaded successfully
- Dataset shape validated
- No missing values detected
- Duplicate datetime records investigated and confirmed as valid daylight saving time overlaps
- No negative electricity demand values detected

The dataset is considered ready for downstream storage and cloud loading.